# Data collection

In [ ]:
pip install mediapipe cvzone opencv-python numpy onnx onnxruntime

# Data collection

In [ ]:
import mediapipe as mp
from cvzone.HandTrackingModule import HandDetector
from cvzone.ClassificationModule import Classifier
import cv2
import numpy as np
import math
import time
import os

# Constants
imgSize = 300
offset = 20
folder = "Data/alphabets/O"
counter = 0

# Create folder if it doesn't exist
os.makedirs(folder, exist_ok=True)

# Initialize webcam and hand detector
cap = cv2.VideoCapture(0)
detector = HandDetector(maxHands=1)

while True:
    success, img = cap.read()
    
    if not success:
        print("Failed to capture image")
        continue  # Skip processing if frame is not captured
    
    hands, img = detector.findHands(img)
    
    if hands:
        hand = hands[0]
        x, y, w, h = hand['bbox']

        # Ensure cropping does not go out of image bounds
        y1, y2 = max(0, y - offset), min(img.shape[0], y + h + offset)
        x1, x2 = max(0, x - offset), min(img.shape[1], x + w + offset)

        imgWhite = np.ones((imgSize, imgSize, 3), np.uint8) * 255
        imgCrop = img[y1:y2, x1:x2]

        # Check if imgCrop is valid before resizing
        if imgCrop.shape[0] > 0 and imgCrop.shape[1] > 0:
            aspectRatio = h / w

            if aspectRatio > 1:
                k = imgSize / h
                wCal = math.ceil(k * w)
                imgResize = cv2.resize(imgCrop, (wCal, imgSize))
                wGap = math.ceil((imgSize - wCal) / 2)
                imgWhite[:, wGap:wCal + wGap] = imgResize
            else:
                k = imgSize / w
                hCal = min(math.ceil(k * h), imgSize)
                imgResize = cv2.resize(imgCrop, (imgSize, hCal))
                hGap = math.ceil((imgSize - hCal) / 2)
                imgWhite[hGap:hCal + hGap, :] = imgResize

            cv2.imshow("ImageCrop", imgCrop)
            cv2.imshow("Image White", imgWhite)

    cv2.imshow("Image", img)
    key = cv2.waitKey(1)

    # Save image when 's' is pressed
    if key == ord('s'):
        counter += 1
        filename = f'{folder}/Image_{time.time()}.jpg'
        if cv2.imwrite(filename, imgWhite):
            print(f"Image saved: {filename}")
        else:
            print("Error: Image not saved!")

    # Exit when 'Esc' key is pressed or window is closed
    if key == 27 or cv2.getWindowProperty("Image", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()


# 

# Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Constants
IMG_SIZE = 224  # Resize images to 224x224
BATCH_SIZE = 32
DATASET_PATH = "Data/alphabets" 
# EPOCHS = 10
# LEARNING_RATE = 0.001
EPOCHS = 20  # Increased for better training
LEARNING_RATE = 0.0005  # Lowered for better fine-tuning


# Image transformations (Data Augmentation + Normalization)

# transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomRotation(20),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=20, translate=(0.1, 0.1)),  # Rotation + Translation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Adjust brightness/contrast
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# Load Dataset
train_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Number of classes
num_classes = len(train_dataset.classes)
print(f"Classes: {train_dataset.classes}")  

# Load Pretrained MobileNetV2
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(in_features=1280, out_features=num_classes)

# Move model to GPU (if available)
model = model.to(device)

# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

# Save the trained model
torch.save(model.state_dict(), "sign_language_model.pth")
print("Model saved successfully!")


# ONNX Conversion

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Constants
IMG_SIZE = 224
num_classes = 24

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the model architecture (same as training)
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(in_features=1280, out_features=num_classes)

# Load the trained weights
model.load_state_dict(torch.load("sign_language_model.pth", map_location=device))
model = model.to(device)
model.eval()  # Set to evaluation mode

# Create a dummy input tensor (batch_size=1, channels=3, height=224, width=224)
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

# Export to ONNX
onnx_path = "sign_language_model.onnx"
torch.onnx.export(
    model,                          # Model to export
    dummy_input,                    # Example input tensor
    onnx_path,                      # Output file path
    export_params=True,             # Store trained weights
    opset_version=11,               # ONNX version
    do_constant_folding=True,       # Optimize constant folding
    input_names=['input'],          # Input tensor names
    output_names=['output'],        # Output tensor names
    dynamic_axes={
        'input': {0: 'batch_size'},    # Variable batch size
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully converted to ONNX format: {onnx_path}")

# Verify the ONNX model
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model verification passed!")

# LLM prompt

In [ ]:
def generate_llm_prompt(str):
    str = "".join(str.split())
    return f"""Correct the spellings and segment the following string: {str}

NOTE:
- Only return the final corrected string without additional information.
- If you have a confusion if the string you're generating is correct or not, then dont' worry just provide any string that you like.
- Only change the spellings only if required, don't change it from one synonym to another.

Sometimes wrongly entered letters present in the input string:
- G sometimes wrongly given as H
- I as D
- L as K or Q
- N as M
- O as C
- R as K
- T as A
- U as K
- V as H
- X as D
- Y as P

For Example if the input is 'HEKKKOWCRLKD' or 'HELCWCRLD' then the output should be "Hello World"

Just provide the output string, don't provide any explanation on how the output is formed or anything

"""

# tkinter gui: using pytorch

In [ ]:
import tkinter as tk
from tkinter import ttk, font
import cv2
import PIL.Image, PIL.ImageTk
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import math
import time
from cvzone.HandTrackingModule import HandDetector
import threading
import requests
import json

class SignLanguageApp:
    def __init__(self, window, window_title):
        # Initialize window
        self.window = window
        self.window.title(window_title)
        self.window.configure(bg='#f0f0f0')
        
        # Constants
        self.IMG_SIZE = 224
        self.OFFSET = 20
        self.MODEL_PATH = "sign_language_model.pth"
        self.CONFIDENCE_THRESHOLD = 80  # Minimum confidence to accept prediction
        self.STABLE_DURATION = 1.5  # Time in seconds a sign must be stable to be accepted
        self.COOLDOWN_PERIOD = 1.0  # Time in seconds before accepting the same letter again
        
        # Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
        # Load the model
        try:
            self.model = models.mobilenet_v2(pretrained=False)
            num_classes = 24
            self.model.classifier[1] = nn.Linear(in_features=1280, out_features=num_classes)
            self.model.load_state_dict(torch.load(self.MODEL_PATH, map_location=self.device))
            self.model.to(self.device)
            self.model.eval()
            
            # Define transformations
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((self.IMG_SIZE, self.IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            
            # ASL class names
            self.class_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
            
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {e}")
            self.model = None
        
        # Initialize variables for sign tracking
        self.current_sign = ""
        self.current_sign_start_time = 0
        self.last_added_sign_time = 0
        self.current_string = ""
        self.last_letter_added = ""
        self.prediction_history = []
        self.history_max_length = 10
        self.prediction = "No hand detected"
        self.confidence = 0.0
        self.llm_output = ""
        self.processing = False
        
        # Initialize webcam
        self.vid = cv2.VideoCapture(0)
        if not self.vid.isOpened():
            raise ValueError("Unable to open video source")
        
        # Initialize hand detector
        self.detector = HandDetector(maxHands=1)
        
        # Create UI layout
        self.create_ui()
        
        # Start video loop
        self.delay = 15
        self.update()
        
        # Bind key events
        self.window.bind('<Escape>', lambda e: self.exit_app())
        self.window.bind('r', lambda e: self.reset_text())
        self.window.bind('m', lambda e: self.process_with_llm())
        
        self.window.protocol("WM_DELETE_WINDOW", self.exit_app)
        self.window.mainloop()
    
    def create_ui(self):
        # Configure grid layout
        self.window.columnconfigure(0, weight=1)
        self.window.columnconfigure(1, weight=1)
        self.window.rowconfigure(0, weight=1)
        self.window.rowconfigure(1, weight=1)
        
        # Define custom fonts
        title_font = font.Font(family="Helvetica", size=12, weight="bold")
        text_font = font.Font(family="Helvetica", size=10)
        
        # 1. Video Frame
        self.video_frame = ttk.LabelFrame(self.window, text="Webcam Feed")
        self.video_frame.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")
        
        self.canvas = tk.Canvas(self.video_frame, width=640, height=480)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # 2. Prediction Frame
        self.prediction_frame = ttk.LabelFrame(self.window, text="Current Prediction")
        self.prediction_frame.grid(row=0, column=1, padx=10, pady=10, sticky="nsew")
        
        self.prediction_label = tk.Label(self.prediction_frame, text="Sign: None", font=text_font)
        self.prediction_label.pack(pady=5)
        
        self.confidence_label = tk.Label(self.prediction_frame, text="Confidence: 0.00%", font=text_font)
        self.confidence_label.pack(pady=5)
        
        self.confidence_bar = ttk.Progressbar(self.prediction_frame, orient="horizontal", length=200, mode="determinate")
        self.confidence_bar.pack(pady=10)
        
        # 3. String Frame
        self.string_frame = ttk.LabelFrame(self.window, text="Formed String")
        self.string_frame.grid(row=1, column=0, padx=10, pady=10, sticky="nsew")
        
        self.string_text = tk.Text(self.string_frame, height=5, width=50, font=text_font, wrap=tk.WORD)
        self.string_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Reset button
        self.reset_button = ttk.Button(self.string_frame, text="Reset (r)", command=self.reset_text)
        self.reset_button.pack(side=tk.LEFT, padx=5, pady=5)
        
        # Process button
        self.process_button = ttk.Button(self.string_frame, text="Process with LLM (m)", 
                                          command=self.process_with_llm)
        self.process_button.pack(side=tk.RIGHT, padx=5, pady=5)
        
        # 4. llm Output Frame
        self.llm_frame = ttk.LabelFrame(self.window, text="LLM Output")
        self.llm_frame.grid(row=1, column=1, padx=10, pady=10, sticky="nsew")
        
        self.llm_text = tk.Text(self.llm_frame, height=5, width=50, font=text_font, wrap=tk.WORD, bg="#f8f8f8")
        self.llm_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Status bar
        self.status_bar = tk.Label(self.window, text="Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM", 
                                   bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.grid(row=2, column=0, columnspan=2, sticky="ew")
    
    def get_stable_prediction(self, history, min_repeats=5):
        """Return the most stable prediction if it appears at least min_repeats times in a row"""
        if len(history) < min_repeats:
            return None, 0
        
        # Check the most recent predictions
        recent = history[-min_repeats:]
        if all(x == recent[0] for x in recent):
            return recent[0], history.count(recent[0])
        return None, 0
    
    def update(self):
        # Read from webcam
        ret, frame = self.vid.read()
        
        if ret:
            # Find hands
            hands, frame = self.detector.findHands(frame)
            
            # Current time
            current_time = time.time()
            
            # Reset prediction if no hand detected
            if not hands:
                self.prediction = "No hand detected"
                self.confidence = 0.0
                self.prediction_history = []
            else:
                # Process hand if detected
                hand = hands[0]
                x, y, w, h = hand['bbox']
                
                # Ensure cropping doesn't go out of bounds
                y1, y2 = max(0, y - self.OFFSET), min(frame.shape[0], y + h + self.OFFSET)
                x1, x2 = max(0, x - self.OFFSET), min(frame.shape[1], x + w + self.OFFSET)
                
                imgWhite = np.ones((self.IMG_SIZE, self.IMG_SIZE, 3), np.uint8) * 255
                imgCrop = frame[y1:y2, x1:x2]
                
                if imgCrop.shape[0] > 0 and imgCrop.shape[1] > 0:
                    aspectRatio = h / w
                    
                    if aspectRatio > 1:
                        k = self.IMG_SIZE / h
                        wCal = math.ceil(k * w)
                        imgResize = cv2.resize(imgCrop, (wCal, self.IMG_SIZE))
                        wGap = math.ceil((self.IMG_SIZE - wCal) / 2)
                        imgWhite[:, wGap:wCal + wGap] = imgResize
                    else:
                        k = self.IMG_SIZE / w
                        hCal = min(math.ceil(k * h), self.IMG_SIZE)
                        imgResize = cv2.resize(imgCrop, (self.IMG_SIZE, hCal))
                        hGap = math.ceil((self.IMG_SIZE - hCal) / 2)
                        imgWhite[hGap:hCal + hGap, :] = imgResize
                    
                    if self.model is not None:
                        # Predict sign
                        input_tensor = self.transform(imgWhite)
                        input_batch = input_tensor.unsqueeze(0).to(self.device)
                        
                        with torch.no_grad():
                            outputs = self.model(input_batch)
                            probabilities = torch.nn.functional.softmax(outputs, dim=1)
                            confidence, predicted = torch.max(probabilities, 1)
                            self.prediction = self.class_names[predicted.item()]
                            self.confidence = confidence.item() * 100
                        
                        # Add prediction to history if confidence is high enough
                        if self.confidence >= self.CONFIDENCE_THRESHOLD:
                            self.prediction_history.append(self.prediction)
                            # Keep history at maximum length
                            if len(self.prediction_history) > self.history_max_length:
                                self.prediction_history.pop(0)
                            
                            # Check for stable prediction
                            stable_pred, stability_count = self.get_stable_prediction(self.prediction_history)
                            
                            # If prediction is stable and different from current sign, start tracking
                            if stable_pred and stable_pred != self.current_sign:
                                self.current_sign = stable_pred
                                self.current_sign_start_time = current_time
                            
                            # If the current sign has been stable for the required duration
                            if (self.current_sign and 
                                stable_pred == self.current_sign and 
                                current_time - self.current_sign_start_time >= self.STABLE_DURATION and
                                (self.current_sign != self.last_letter_added or 
                                 current_time - self.last_added_sign_time >= self.COOLDOWN_PERIOD)):
                                
                                # Add the letter to our string
                                if self.current_string:  # If not empty, add a space
                                    self.current_string += " " + self.current_sign
                                else:
                                    self.current_string = self.current_sign
                                
                                # Update string display
                                self.string_text.delete(1.0, tk.END)
                                self.string_text.insert(tk.END, self.current_string)
                                
                                # Update tracking variables
                                self.last_letter_added = self.current_sign
                                self.last_added_sign_time = current_time
                                
                                # Reset history to avoid immediate repetition
                                self.prediction_history = []
                                self.current_sign = ""
            
            # Convert frame for display
            cv2image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            current_image = PIL.Image.fromarray(cv2image)
            
            # Resize to fit canvas
            canvas_width = self.canvas.winfo_width()
            canvas_height = self.canvas.winfo_height()
            
            if canvas_width > 1 and canvas_height > 1:  # Ensure canvas has size
                current_image = current_image.resize((canvas_width, canvas_height), PIL.Image.LANCZOS)
            
            # Convert to PhotoImage
            self.photo = PIL.ImageTk.PhotoImage(image=current_image)
            
            # Display the image
            self.canvas.create_image(0, 0, image=self.photo, anchor=tk.NW)
            
            # Update prediction display
            self.prediction_label.config(text=f"Sign: {self.prediction}")
            self.confidence_label.config(text=f"Confidence: {self.confidence:.2f}%")
            self.confidence_bar["value"] = min(self.confidence, 100)
        
        # Schedule next update
        self.window.after(self.delay, self.update)
    
    def reset_text(self):
        self.current_string = ""
        self.string_text.delete(1.0, tk.END)
        self.llm_text.delete(1.0, tk.END)
        self.llm_output = ""
        self.prediction_history = []
        self.current_sign = ""
        self.status_bar.config(text="Text reset | Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM")
    
    def process_with_llm(self):
        # Check if there's text to process
        if not self.current_string or self.processing:
            return
        
        self.processing = True
        self.status_bar.config(text="Processing with LLM...")
        
        # Start processing in a separate thread to avoid UI freeze
        threading.Thread(target=self.call_llm_api).start()
    
    def call_LLM_api(self):
        try:
            # Call Ollama API (llm model)
            url = "http://localhost:11434/api/generate"
            payload = {
                "model": "mistral",
                "prompt" : generate_llm_prompt(self.current_string),
                "stream": False
            }
            
            response = requests.post(url, json=payload)
            
            if response.status_code == 200:
                result = response.json()
                self.llm_output = result.get("response", "No response from LLM")
            else:
                self.llm_output = f"Error: {response.status_code} - {response.text}"
            
            # Update UI from main thread
            self.window.after(0, self.update_llm_output)
        except Exception as e:
            self.llm_output = f"Error processing with LLM: {str(e)}"
            self.window.after(0, self.update_llm_output)
        
        self.processing = False
    
    def update_llm_output(self):
        self.llm_text.delete(1.0, tk.END)
        self.llm_text.insert(tk.END, self.llm_output)
        self.status_bar.config(text="Processed with LLM | Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM")
    
    def exit_app(self):
        # Release resources
        if self.vid.isOpened():
            self.vid.release()
        self.window.destroy()

# Run application
if __name__ == "__main__":
    root = tk.Tk()
    app = SignLanguageApp(root, "Sign Language Recognition")

#  Inference Using ONNX

In [ ]:
import tkinter as tk
from tkinter import ttk, font
import cv2
import PIL.Image, PIL.ImageTk
import numpy as np
import math
import time
from cvzone.HandTrackingModule import HandDetector
import threading
import requests
import onnxruntime as ort

class SignLanguageApp:
    def __init__(self, window, window_title):
        # Initialize window
        self.window = window
        self.window.title(window_title)
        self.window.configure(bg='#f0f0f0')
        
        # Constants
        self.IMG_SIZE = 224
        self.OFFSET = 20
        self.MODEL_PATH = "sign_language_model.onnx"
        self.CONFIDENCE_THRESHOLD = 80  # Minimum confidence to accept prediction
        self.STABLE_DURATION = 1.5  # Time in seconds a sign must be stable to be accepted
        self.COOLDOWN_PERIOD = 1.0  # Time in seconds before accepting the same letter again
        
        # Load the ONNX model
        try:
            self.session = ort.InferenceSession(self.MODEL_PATH)
            self.input_name = self.session.get_inputs()[0].name
            self.output_name = self.session.get_outputs()[0].name
            
            # ASL class names
            self.class_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
            
            print("ONNX Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {e}")
            self.session = None
        
        # Initialize variables for sign tracking
        self.current_sign = ""
        self.current_sign_start_time = 0
        self.last_added_sign_time = 0
        self.current_string = ""
        self.last_letter_added = ""
        self.prediction_history = []
        self.history_max_length = 10
        self.prediction = "No hand detected"
        self.confidence = 0.0
        self.inference_time = 0.0
        self.llm_output = ""
        self.processing = False
        
        # Initialize webcam
        self.vid = cv2.VideoCapture(0)
        if not self.vid.isOpened():
            raise ValueError("Unable to open video source")
        
        # Initialize hand detector
        self.detector = HandDetector(maxHands=1)
        
        # Create UI layout
        self.create_ui()
        
        # Start video loop
        self.delay = 15
        self.update()
        
        # Bind key events
        self.window.bind('<Escape>', lambda e: self.exit_app())
        self.window.bind('r', lambda e: self.reset_text())
        self.window.bind('m', lambda e: self.process_with_llm())
        
        self.window.protocol("WM_DELETE_WINDOW", self.exit_app)
        self.window.mainloop()
    
    def create_ui(self):
        # Configure grid layout
        self.window.columnconfigure(0, weight=1)
        self.window.columnconfigure(1, weight=1)
        self.window.rowconfigure(0, weight=1)
        self.window.rowconfigure(1, weight=1)
        
        # Define custom fonts
        title_font = font.Font(family="Helvetica", size=12, weight="bold")
        text_font = font.Font(family="Helvetica", size=10)
        
        # 1. Video Frame
        self.video_frame = ttk.LabelFrame(self.window, text="Webcam Feed")
        self.video_frame.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")
        
        self.canvas = tk.Canvas(self.video_frame, width=640, height=480)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # 2. Prediction Frame
        self.prediction_frame = ttk.LabelFrame(self.window, text="Current Prediction")
        self.prediction_frame.grid(row=0, column=1, padx=10, pady=10, sticky="nsew")
        
        self.prediction_label = tk.Label(self.prediction_frame, text="Sign: None", font=text_font)
        self.prediction_label.pack(pady=5)
        
        self.confidence_label = tk.Label(self.prediction_frame, text="Confidence: 0.00%", font=text_font)
        self.confidence_label.pack(pady=5)
        
        self.confidence_bar = ttk.Progressbar(self.prediction_frame, orient="horizontal", length=200, mode="determinate")
        self.confidence_bar.pack(pady=10)
        
        # Inference time label
        self.inference_time_label = tk.Label(self.prediction_frame, text="Inference Time: 0.00 ms", 
                                             font=text_font, fg="#0066cc")
        self.inference_time_label.pack(pady=5)
        
        # 3. String Frame
        self.string_frame = ttk.LabelFrame(self.window, text="Formed String")
        self.string_frame.grid(row=1, column=0, padx=10, pady=10, sticky="nsew")
        
        self.string_text = tk.Text(self.string_frame, height=5, width=50, font=text_font, wrap=tk.WORD)
        self.string_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Reset button
        self.reset_button = ttk.Button(self.string_frame, text="Reset (r)", command=self.reset_text)
        self.reset_button.pack(side=tk.LEFT, padx=5, pady=5)
        
        # Process button
        self.process_button = ttk.Button(self.string_frame, text="Process with LLM (m)", 
                                          command=self.process_with_llm)
        self.process_button.pack(side=tk.RIGHT, padx=5, pady=5)
        
        # 4. LLM Output Frame
        self.llm_frame = ttk.LabelFrame(self.window, text="LLM Output")
        self.llm_frame.grid(row=1, column=1, padx=10, pady=10, sticky="nsew")
        
        self.llm_text = tk.Text(self.llm_frame, height=5, width=50, font=text_font, wrap=tk.WORD, bg="#f8f8f8")
        self.llm_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Status bar
        self.status_bar = tk.Label(self.window, text="Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM", 
                                   bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.grid(row=2, column=0, columnspan=2, sticky="ew")
    
    def preprocess_image(self, img):
        """Preprocess image for ONNX model"""
        # Resize
        img = cv2.resize(img, (self.IMG_SIZE, self.IMG_SIZE))
        
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Normalize to [0, 1]
        img = img.astype(np.float32) / 255.0
        
        # Normalize with ImageNet mean and std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        
        # HWC to CHW format
        img = np.transpose(img, (2, 0, 1))
        
        # Add batch dimension
        img = np.expand_dims(img, axis=0)
        
        return img
    
    def softmax(self, x):
        """Compute softmax values for array x"""
        exp_x = np.exp(x - np.max(x))
        return exp_x / exp_x.sum(axis=1, keepdims=True)
    
    def get_stable_prediction(self, history, min_repeats=5):
        """Return the most stable prediction if it appears at least min_repeats times in a row"""
        if len(history) < min_repeats:
            return None, 0
        
        # Check the most recent predictions
        recent = history[-min_repeats:]
        if all(x == recent[0] for x in recent):
            return recent[0], history.count(recent[0])
        return None, 0
    
    def update(self):
        # Read from webcam
        ret, frame = self.vid.read()
        
        if ret:
            # Find hands
            hands, frame = self.detector.findHands(frame)
            
            # Current time
            current_time = time.time()
            
            # Reset prediction if no hand detected
            if not hands:
                self.prediction = "No hand detected"
                self.confidence = 0.0
                self.inference_time = 0.0
                self.prediction_history = []
            else:
                # Process hand if detected
                hand = hands[0]
                x, y, w, h = hand['bbox']
                
                # Ensure cropping doesn't go out of bounds
                y1, y2 = max(0, y - self.OFFSET), min(frame.shape[0], y + h + self.OFFSET)
                x1, x2 = max(0, x - self.OFFSET), min(frame.shape[1], x + w + self.OFFSET)
                
                imgWhite = np.ones((self.IMG_SIZE, self.IMG_SIZE, 3), np.uint8) * 255
                imgCrop = frame[y1:y2, x1:x2]
                
                if imgCrop.shape[0] > 0 and imgCrop.shape[1] > 0:
                    aspectRatio = h / w
                    
                    if aspectRatio > 1:
                        k = self.IMG_SIZE / h
                        wCal = math.ceil(k * w)
                        imgResize = cv2.resize(imgCrop, (wCal, self.IMG_SIZE))
                        wGap = math.ceil((self.IMG_SIZE - wCal) / 2)
                        imgWhite[:, wGap:wCal + wGap] = imgResize
                    else:
                        k = self.IMG_SIZE / w
                        hCal = min(math.ceil(k * h), self.IMG_SIZE)
                        imgResize = cv2.resize(imgCrop, (self.IMG_SIZE, hCal))
                        hGap = math.ceil((self.IMG_SIZE - hCal) / 2)
                        imgWhite[hGap:hCal + hGap, :] = imgResize
                    
                    if self.session is not None:
                        # Preprocess image
                        input_tensor = self.preprocess_image(imgWhite)
                        
                        # Measure inference time
                        start_time = time.perf_counter()
                        
                        # Run inference
                        outputs = self.session.run([self.output_name], {self.input_name: input_tensor})
                        
                        end_time = time.perf_counter()
                        self.inference_time = (end_time - start_time) * 1000  # Convert to milliseconds
                        
                        # Get prediction
                        probabilities = self.softmax(outputs[0])
                        predicted_idx = np.argmax(probabilities)
                        self.confidence = probabilities[0][predicted_idx] * 100
                        self.prediction = self.class_names[predicted_idx]
                        
                        # Add prediction to history if confidence is high enough
                        if self.confidence >= self.CONFIDENCE_THRESHOLD:
                            self.prediction_history.append(self.prediction)
                            # Keep history at maximum length
                            if len(self.prediction_history) > self.history_max_length:
                                self.prediction_history.pop(0)
                            
                            # Check for stable prediction
                            stable_pred, stability_count = self.get_stable_prediction(self.prediction_history)
                            
                            # If prediction is stable and different from current sign, start tracking
                            if stable_pred and stable_pred != self.current_sign:
                                self.current_sign = stable_pred
                                self.current_sign_start_time = current_time
                            
                            # If the current sign has been stable for the required duration
                            if (self.current_sign and 
                                stable_pred == self.current_sign and 
                                current_time - self.current_sign_start_time >= self.STABLE_DURATION and
                                (self.current_sign != self.last_letter_added or 
                                 current_time - self.last_added_sign_time >= self.COOLDOWN_PERIOD)):
                                
                                # Add the letter to our string
                                if self.current_string:  # If not empty, add a space
                                    self.current_string += " " + self.current_sign
                                else:
                                    self.current_string = self.current_sign
                                
                                # Update string display
                                self.string_text.delete(1.0, tk.END)
                                self.string_text.insert(tk.END, self.current_string)
                                
                                # Update tracking variables
                                self.last_letter_added = self.current_sign
                                self.last_added_sign_time = current_time
                                
                                # Reset history to avoid immediate repetition
                                self.prediction_history = []
                                self.current_sign = ""
            
            # Convert frame for display
            cv2image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            current_image = PIL.Image.fromarray(cv2image)
            
            # Resize to fit canvas
            canvas_width = self.canvas.winfo_width()
            canvas_height = self.canvas.winfo_height()
            
            if canvas_width > 1 and canvas_height > 1:  # Ensure canvas has size
                current_image = current_image.resize((canvas_width, canvas_height), PIL.Image.LANCZOS)
            
            # Convert to PhotoImage
            self.photo = PIL.ImageTk.PhotoImage(image=current_image)
            
            # Display the image
            self.canvas.create_image(0, 0, image=self.photo, anchor=tk.NW)
            
            # Update prediction display
            self.prediction_label.config(text=f"Sign: {self.prediction}")
            self.confidence_label.config(text=f"Confidence: {self.confidence:.2f}%")
            self.confidence_bar["value"] = min(self.confidence, 100)
            self.inference_time_label.config(text=f"Inference Time: {self.inference_time:.2f} ms")
        
        # Schedule next update
        self.window.after(self.delay, self.update)
    
    def reset_text(self):
        self.current_string = ""
        self.string_text.delete(1.0, tk.END)
        self.llm_text.delete(1.0, tk.END)
        self.llm_output = ""
        self.prediction_history = []
        self.current_sign = ""
        self.status_bar.config(text="Text reset | Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM")
    
    def process_with_llm(self):
        # Check if there's text to process
        if not self.current_string or self.processing:
            return
        
        self.processing = True
        self.status_bar.config(text="Processing with LLM...")
        
        # Start processing in a separate thread to avoid UI freeze
        threading.Thread(target=self.call_llm_api, daemon=True).start()
    
    def generate_llm_prompt(self, text):
        """Generate prompt for LLM"""
        return f"""The following text was captured from sign language recognition. 
It may contain spelling errors or missing spaces. Please correct it and provide 
a properly formatted version:

Input: {text}

Corrected text:"""
    
    def call_llm_api(self):
        try:
            # Call Ollama API (LLM model)
            url = "http://localhost:11434/api/generate"
            payload = {
                "model": "mistral",
                "prompt": self.generate_llm_prompt(self.current_string),
                "stream": False
            }
            
            response = requests.post(url, json=payload, timeout=30)
            
            if response.status_code == 200:
                result = response.json()
                self.llm_output = result.get("response", "No response from LLM")
            else:
                self.llm_output = f"Error: {response.status_code} - {response.text}"
            
            # Update UI from main thread
            self.window.after(0, self.update_llm_output)
        except Exception as e:
            self.llm_output = f"Error processing with LLM: {str(e)}"
            self.window.after(0, self.update_llm_output)
        
        self.processing = False
    
    def update_llm_output(self):
        self.llm_text.delete(1.0, tk.END)
        self.llm_text.insert(tk.END, self.llm_output)
        self.status_bar.config(text="Processed with LLM | Press 'ESC' to exit | 'r' to reset | 'm' to process with LLM")
    
    def exit_app(self):
        # Release resources
        if self.vid.isOpened():
            self.vid.release()
        self.window.destroy()

# Run application
if __name__ == "__main__":
    root = tk.Tk()
    app = SignLanguageApp(root, "Sign Language Recognition (ONNX)")